In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

status_history = spark.table(
    "logistics.silver.silver_shipment_status_history"
)

display(status_history)

In [0]:
status_history.printSchema()

In [0]:
lifecycle_window = Window.partitionBy(
    "shipment_id"
).orderBy(
    col("event_ts").asc(),
    col("received_ts").asc_nulls_last()
)

lifecycle_df = status_history.withColumn(
    "lifecycle_sequence",
    row_number().over(lifecycle_window)
)

display(
    lifecycle_df.orderBy(
        "shipment_id",
        "lifecycle_sequence"
    )
)

In [0]:
from pyspark.sql.functions import lag, lead

lifecycle_df = lifecycle_df.withColumn(
    "previous_event",
    lag("event_type").over(lifecycle_window)
)

lifecycle_df = lifecycle_df.withColumn(
    "next_event",
    lead("event_type").over(lifecycle_window)
)

display(lifecycle_df)

In [0]:
lifecycle_df = lifecycle_df.withColumn(
    "transition_status",
    when(
        col("previous_event") == "DELIVERED",
        "INVALID"
    ).otherwise("VALID_OR_UNCHECKED")
)

display(lifecycle_df)

In [0]:
valid_lifecycle = lifecycle_df.filter(col("transition_status") != "INVALID")

lifecycle_exceptions = lifecycle_df.filter(col("transition_status") == "INVALID").withColumn(
    "exception_reason",
    lit("Shipment has another event after DELIVERED")
)

In [0]:
lifecycle_exceptions = lifecycle_exceptions.withColumn(
    "exception_reason",
    lit("Shipment has another event after DELIVERED")
)

In [0]:
print("Total lifecycle events:", lifecycle_df.count())

print("Valid or unchecked events:", valid_lifecycle.count())

print("Lifecycle exceptions:", lifecycle_exceptions.count())

In [0]:
lifecycle_df = lifecycle_df.withColumn(
    "next_event_ts",
    lead("event_ts").over(lifecycle_window)
)

lifecycle_df = lifecycle_df.withColumn(
    "stage_duration_hours",
    (
        unix_timestamp("next_event_ts") -
        unix_timestamp("event_ts")
    ) / 3600
)

display(
    lifecycle_df.select(
        "shipment_id",
        "event_type",
        "event_ts",
        "next_event",
        "next_event_ts",
        "stage_duration_hours"
    )
)

In [0]:
shipment_summary = lifecycle_df.groupBy(
    "shipment_id"
).agg(
    count("*").alias("total_events"),

    min(
        when(
            col("event_type") == "PICKED_UP",
            col("event_ts")
        )
    ).alias("pickup_time"),

    max(
        when(
            col("event_type") == "DELIVERED",
            col("event_ts")
        )
    ).alias("delivery_time")
)

shipment_summary = shipment_summary.withColumn(
    "delivery_duration_hours",
    (
        unix_timestamp("delivery_time") -
        unix_timestamp("pickup_time")
    ) / 3600
)

shipment_summary = shipment_summary.withColumn(
    "delivery_observed",
    when(
        col("delivery_time").isNotNull(),
        "YES"
    ).otherwise("NO")
)

display(shipment_summary)

In [0]:
lifecycle_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "logistics.silver.silver_shipment_lifecycle"
    )

In [0]:
lifecycle_exceptions.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "logistics.silver.silver_lifecycle_exceptions"
    )

In [0]:
shipment_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "logistics.silver.silver_shipment_lifecycle_summary"
    )

for t in ["silver_shipment_lifecycle", "silver_lifecycle_exceptions", "silver_shipment_lifecycle_summary"]:
    print(t, spark.table(f"logistics.silver.{t}").count())

In [0]:
%sql
SELECT event_type, COUNT(*)
FROM logistics.silver.silver_shipment_lifecycle
GROUP BY event_type;

In [0]:
%sql
SELECT *
FROM logistics.silver.silver_lifecycle_exceptions;

In [0]:
%sql
SELECT *
FROM logistics.silver.silver_shipment_lifecycle_summary
ORDER BY delivery_duration_hours DESC;

In [0]:
%sql
SELECT *
FROM logistics.silver.silver_shipment_lifecycle_summary
WHERE delivery_observed = 'NO';